In [1]:
import os
import pandas as pd
import numpy as np
from collections import defaultdict


In [2]:
base_dir = os.path.join('/Users/lukelee/Movies/Boxing/')

In [78]:
df = pd.read_csv(os.path.join(base_dir,'audio_time_stamp_full.csv'),index_col=0)

df['Delta']=df['Delta'].fillna(0)

df.loc[:,'half_min'] = list(map(int,np.round(df['Delta']/30,0)))
df.loc[:,'Bell_Details'] = ''
df.loc[:,'Round'] = 0

config_dict = defaultdict(int)


for i,v in df.loc[df['Type']=='Warning_Bell'].iterrows():
    before_type = df.loc[i-1,'Type']
    after_type = df.loc[i+1,'Type']

    if before_type == 'Bell' and df.loc[i,'half_min'] in [3,5]:
        df.loc[i-1,'Bell_Details'] = 'Start'
        #df.loc[i-1,'Round'] = 1

    if after_type == 'Bell' and df.loc[i+1,'half_min'] == 1:
        df.loc[i+1,'Bell_Details'] = 'End'

    df.loc[i,'Bell_Details'] = 'Warning'

    config_key = "_".join(map(str,df.loc[i-1:i+1,'half_min'].to_list()))
    config_dict[config_key] += 1
#df.loc[:,'Round'] = df['Round'].cumsum()
final_config = max(config_dict, key=config_dict.get)
config_df = pd.DataFrame()
config_df.loc[:,'half_min_config'] = list(map(int,final_config.split('_')))
config_df.loc[:,'Bell_Details'] = ('Start','Warning','End')
df = pd.merge(df,config_df,how='left',on='Bell_Details')
df.loc[:,'Diff'] = df['half_min_config']-df['half_min']

In [72]:
df.loc[df['Diff'] < 0,:]

,Type,Seconds,Formatted_Time,Delta,half_min,Bell_Details,Round,half_min_config,Diff
4,Bell,186,00:03:06,62.0,2,Start,0,1.0,-1.0
22,Warning_Bell,1174,00:19:34,119.0,4,Warning,0,3.0,-1.0
26,Bell,1384,00:23:04,60.0,2,Start,0,1.0,-1.0


In [79]:
# if bell before Warning_Bell and if delta is 1.5min or 2.5 min then bell is Start Bell
# if bell after Warning_Bell and if delta is 30, then bell is End Bell
# A round consists of Start Bell, Warning_Bell, End_Bell
# After End_Bell, either 30 sec or 60 sec for Start Bell

for i,v in df.loc[df['Diff'] < 0,:].iterrows():
    curr_bell = v['Bell_Details']
    curr_second = v['Seconds']
    diff = v['Diff']
    new_slice = pd.DataFrame()
    new_type = ''
    if curr_bell == 'Warning':
        # adding a missing Start Bell
        new_type='Start'
    elif curr_bell == 'Start':
        new_type='End'
    elif curr_bell == 'End':
        new_type = 'Warning'

    new_slice = pd.DataFrame({'Type':['Fixed_Bell'],'Seconds':[curr_second+diff*30],'Delta':[-1*diff*30],'Bell_Details':[new_type]})

    # Fix curr slice
    df.loc[i,'Delta'] += diff*30
    df.loc[i,'half_min'] += diff
    df.loc[i,'Diff'] = 0

    # insert by adding to end
    df = pd.concat([df,new_slice])

# sort by Seconds to fix index
df = df.sort_values('Seconds').reset_index(drop=True)
df = df.drop(columns=['half_min','half_min_config','Diff'])

df = df.loc[df.loc[df['Bell_Details']=='Start',:].index.min():df.loc[df['Bell_Details']=='End',:].index.max(),]



In [80]:
df.loc[df['Bell_Details']=='Start','Round'] = 1

In [81]:
df.loc[:,'Round'] = df['Round'].cumsum()

In [93]:
df.loc[df['Formatted_Time'].isna(),:].index.to_list()

[23, 28]

In [66]:
df.head(10)

,Type,Seconds,Formatted_Time,Delta,Bell_Details,Round
0,Start,0.0,00:00:00,0.0,,0.0
1,Warning_Bell,63.0,00:01:03,63.0,Warning,0.0
2,Bell,93.0,00:01:33,30.0,End,0.0
3,Bell,124.0,00:02:04,31.0,,0.0
4,Fixed_Bell,156.0,NaN,NaN,End,NaN
5,Bell,186.0,00:03:06,32.0,Start,0.0
6,Warning_Bell,275.0,00:04:35,89.0,Warning,0.0
7,Bell,305.0,00:05:05,30.0,End,0.0
8,Bell,335.0,00:05:35,30.0,Start,0.0
9,Warning_Bell,425.0,00:07:05,90.0,Warning,0.0


,Type,Seconds,Formatted_Time,Delta,half_min,Bell_Details,Round
4,Bell,186,00:03:06,62.0,2.0,Start,0.0
7,Bell,335,00:05:35,30.0,1.0,Start,0.0
10,Bell,485,00:08:05,30.0,1.0,Start,0.0
13,Bell,635,00:10:35,30.0,1.0,Start,0.0
16,Bell,785,00:13:05,30.0,1.0,Start,0.0
19,Bell,935,00:15:35,30.0,1.0,Start,0.0
24,Bell,1234,00:20:34,30.0,1.0,Start,0.0
26,Bell,1384,00:23:04,60.0,2.0,Start,0.0
29,Bell,1534,00:25:34,30.0,1.0,Start,0.0
32,Bell,1684,00:28:04,30.0,1.0,Start,0.0


In [50]:
# from the df -> find the configuration (i.e. rest 30 sec & Round 2 min)

# edit the df for missing bells

,Type,Seconds,Formatted_Time,Delta,half_min,Bell_Details,Round
0,Start,0,00:00:00,NaN,NaN,,0.0
1,Warning_Bell,63,00:01:03,63.0,2.0,,0.0
2,Bell,93,00:01:33,30.0,1.0,End,0.0
3,Bell,124,00:02:04,31.0,1.0,,0.0
4,Bell,186,00:03:06,62.0,2.0,Start,1.0
5,Warning_Bell,275,00:04:35,89.0,3.0,,1.0
6,Bell,305,00:05:05,30.0,1.0,End,1.0
7,Bell,335,00:05:35,30.0,1.0,Start,2.0
8,Warning_Bell,425,00:07:05,90.0,3.0,,2.0
9,Bell,455,00:07:35,30.0,1.0,End,2.0
